In [ ]:
import pandas as pd 
import numpy as np 
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import  make_pipeline


In [ ]:
from catboost import CatBoostClassifier
from sklearn.base import clone
# fix for a catboost bug! -- without this snippet it's impossible to pass the indices for 
# the categorical features to the constructor for the catboost classifier

class CustomCatBoostClassifier(CatBoostClassifier):
    def __sklearn_clone__(self):
        return CustomCatBoostClassifier(**self.get_params())

In [ ]:
dev = pd.read_csv("development(1).csv")
ev = pd.read_csv("evaluation.csv")

y = dev["death"]
x = dev.drop(columns=["death"])
cat_indices = [i for i, _ in enumerate(x.drop(columns=["temp_f", "Id", "adls"])) if _ in {"sex", "medical_center", "dzgroup", "dzclass", 
                                                           "race", "ca", "dnr"}]

In [ ]:
from feature_eng import pipe_non_tree_new_eng, pipe_random_tree_new_eng, pipe_boosted_tree_new_neg
features_lgbm =  {'n_estimators': 3342, 'learning_rate': 0.0028954614575091887, 'min_data_in_leaf': 81, 
                          'max_depth': 9, 'min_gain_to_split': 1.3209989283293588, 
                          'subsample': 0.8822893771001694, 'subsample_freq': 2, 
                          'max_bin': 238, 'feature_fraction': 0.6511238486969437, 
                          'reg_lambda': 0.04227276364865463, 
                          'reg_alpha': 0.8086097224250781, 
                          'scale_pos_weight': 0.49326982842529654, 
                          'num_leaves': 198, "random_state": 17} # 0.4653740340963909 0.7576285347683017
features_random_forest = {'max_features': 'log2', 'max_depth': 28, 'min_samples_leaf': 3, 'n_estimators': 4651, 'class_weight': 'balanced', "random_state": 17, "n_jobs": -1} 
features_cat_boost = {'iterations': 4174, 'learning_rate': 0.0040932218353216246, 'random_strength': 0.001562384211665802, 'depth': 6, 
'bagging_temperature': 0.8156555071877585, 'border_count': 214, 'l2_leaf_reg': 19, 'scale_pos_weight': 0.47919356417193015, "cat_features": cat_indices, "random_state": 17} # 0.7609402917581694 0.45082660405796093
features_xg_boost = {'n_estimators': 4717, 'learning_rate': 0.005221174518290643, 'min_child_weight': 1, 'max_depth': 2, 
'subsample': 0.9099870866208125, 'colsample_bytree': 0.8838338623066402, 
'colsample_bylevel': 0.8534163510022217, 'max_bin': 195, 'reg_lambda': 1.9188323258235047, 
'reg_alpha': 0.00025552830819887515, 'gamma': 0.00019769506514463376, 'scale_pos_weight': 0.39188495395584216, "enable_categorical":True, "tree_method":"hist", "device":"cuda", "random_state":17, "n_jobs": -1} # 0.7577273510746684 0.36024192
features_logistic = {'C': 0.052276215026302975, 'l1_ratio': 0.851844421406245, "max_iter": 5000, "solver": "saga"} # 0.7507453798891423 0.5420956679213134



model_pipe = [
    (0.4653740340963909,"lgbm",make_pipeline(pipe_boosted_tree_new_neg, LGBMClassifier(**features_lgbm))), (0.545837648927173, "rf", make_pipeline(
        pipe_random_tree_new_eng, RandomForestClassifier(**features_random_forest)
    )), (0.45082660405796093,"catboost", make_pipeline(pipe_boosted_tree_new_neg, CustomCatBoostClassifier(**features_cat_boost))), (0.36024192,"xg_boost", make_pipeline(pipe_boosted_tree_new_neg, XGBClassifier(**features_xg_boost))), (0.5420956679213134,"lgr", make_pipeline(pipe_non_tree_new_eng, LogisticRegression(**features_logistic)))
]


In [ ]:
# This will create the submission files for my top 2 submissions! (catboost and xgboost models)
for th, name, model in model_pipe[-3: -1]: 
    model.fit(x, y)
    pd.DataFrame(model.predict_proba(ev)[:,1] >= th, columns=["Predicted"]).to_csv(f"{name}_sub.csv", index_label="Id")

